In [1]:
import matplotlib.pyplot as plt
import math
import matplotlib.patches as patches
import numpy as np
import matplotlib
import matplotlib.animation as anm    
from IPython.display import HTML   

In [2]:
class World:
    def __init__(self,time_span,time_interval,debug=False):    
        self.objects = []   
        self.debug = debug
        self.time_span = time_span    #何秒間シミュレーションするか
        self.time_interval = time_interval    #⊿t

    def append(self,obj):   
        self.objects.append(obj)

    def draw(self):
        fig = plt.figure(figsize=(4,4))    
        ax = fig.add_subplot(111)         
        ax.set_aspect('equal')            
        ax.set_xlim(-5,5)                 
        ax.set_ylim(-5,5)                  
        ax.set_xlabel("X",fontsize=10)     
        ax.set_ylabel("Y",fontsize=10)    

        elems = []  
        
        if self.debug:
            for i in range(1000):self.one_step(i,elems,ax)    
        else:
            self.ani = anm.FuncAnimation(fig,self.one_step,fargs=(elems,ax),
                                         frames=int(self.time_span/self.time_interval)+1,interval=int(self.time_interval*1000),repeat=False)
            plt.close()    
            return HTML(self.ani.to_jshtml()) 

    def one_step(self,i,elems,ax):    
        while elems:elems.pop().remove()  
        time_str = "t = %.2f[s]"%(self.time_interval*i)    #時刻として表示する文字列
        elems.append(ax.text(-4.4, 4.5,"t = "+str(i),fontsize=10))    
        for obj in self.objects:
            obj.draw(ax,elems)
            if hasattr(obj,"one_step"):obj.one_step(self.time_interval)    

In [3]:
class IdealRobot:
    def __init__(self,pose,agent=None,color="black"):    #agentを追加
        self.pose = pose    
        self.r = 0.2        
        self.color = color
        self.agent = agent    #agent追加
        self.poses = [pose]    #軌跡の描画用

    def draw(self,ax,elems):   
        x,y,theta = self.pose    
        xn = x + self.r * math.cos(theta)    
        yn = y + self.r * math.sin(theta)    
        elems += ax.plot([x,xn],[y,yn],color=self.color)    
        c = patches.Circle(xy=(x,y),radius=self.r,fill=False,color=self.color)    
        elems.append(ax.add_patch(c))   

        self.poses.append(self.pose)    #軌跡の描画
        elems += ax.plot([e[0] for e in self.poses],[e[1] for e in self.poses],linewidth=0.5,color="black")

    @classmethod    #オブジェクトを作らなくてもこのメソッドを実行できるようにする　すべてのロボットに共通する部分だから
    def state_transition(cls,nu,omega,time,pose):    #移動先の姿勢を求める
        t0 = pose[2]
        if math.fabs(omega) < 1e-10:    #角速度がほぼ０の場合とそうでない場合に分ける→状態遷移関数が異なる
            return pose + np.array([nu*math.cos(t0),
                                    nu*math.sin(t0),
                                    omega])*time
        else:
            return pose + np.array([nu/omega*(math.sin(t0+omega*time)-math.sin(t0)),
                                    nu/omega*(-math.cos(t0+omega*time)+math.cos(t0)),
                                    omega*time])

    def one_step(self,time_interval):    #エージェントがいる場合、速度と角速度を受け取って姿勢を更新する　time_intervalは離散時間１ステップ分が何秒になるか指定
        if not self.agent:return    #エージェントがいないときそのまま返す
        nu, omega = self.agent.decision()
        self.pose = self.state_transition(nu,omega,time_interval,self.pose)

In [4]:
class Agent:
    def __init__(self,nu,omega):
        self.nu = nu
        self.omega = omega

    def decision(self,observation=None):    #observationはセンサ値の受け渡しに使われる
        return self.nu,self.omega

In [5]:
world = World(10,1)    #コードにバグがあるときworld=World(debug=True)としてデバッグを行う

straight = Agent(0.2,0.0)    #0.2[m/s]で直進
circling = Agent(0.2,10.0/180*math.pi)    #0.2[m/s],10[deg/s]で円を描きながら

robot1 = IdealRobot(np.array([2,3,math.pi/6]).T,straight)    #エージェントstraight
robot2 = IdealRobot(np.array([-2,-1,math.pi/5*6]).T,circling,"red")    #エージェントcircling
robot3 = IdealRobot(np.array([0,0,0]).T,color="blue")    #エージェントを与えないロボット

world.append(robot1)    
world.append(robot2)
world.append(robot3)

world.draw()